In [1]:
import os
import math
import pickle
import shutil
import warnings
from pprint import pprint
from typing import Union, Dict, List, Tuple

import yaml
import boto3
import pandas as pd
import numpy as np
import mlflow
import optuna

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from mlflow.models import infer_signature


warnings.filterwarnings('ignore')


c:\ProgramData\miniconda3\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_yaml_file(path, file):
    with open(os.path.join(path, file)) as f:
        try:
            content = yaml.safe_load(f)
        except yaml.YAMLError as e:
            raise e
    
    return content


CONFIG_PATH = os.path.join("..", "src", "config")

In [3]:
with open("VERSION", "r") as f:
    CODE_VERSION = f.readline().strip()

credentials = read_yaml_file(path=CONFIG_PATH, file="credentials.yaml")
settings = read_yaml_file(path=CONFIG_PATH, file="settings.yaml")

EC2_URL = credentials['EC2_URL']
mlflow.set_tracking_uri(f"http://{EC2_URL}:5000")
print(f"Tracking Server URI: '{mlflow.get_tracking_uri()}'")

SEED = 42
ARTIFACTS_OUTPUT_PATH = settings['ARTIFACTS_PATH']
FEATURES_OUTPUT_PATH = settings['FEATURES_PATH']
RAW_FILE_PATH = os.path.join(settings["DATA_PATH"], settings["RAW_FILE_NAME"])
PROCESSED_RAW_FILE = "Preprocessed_" + settings["RAW_FILE_NAME"]
PROCESSED_RAW_FILE_PATH = os.path.join(settings["DATA_PATH"], PROCESSED_RAW_FILE)
FEATURE_SELECTION_EXPERIMENT_NAME = "feature-selection-experimentation"
HYPERPARAMETER_TUNING_EXPERIMENT_NAME = "hyperparameters-tuning-experimentation"


Tracking Server URI: 'http://3.90.187.253:5000'


In [4]:
RAW_FILE_PATH = f"../{RAW_FILE_PATH}"
PROCESSED_RAW_FILE_PATH = f"../{PROCESSED_RAW_FILE_PATH}"
ARTIFACTS_OUTPUT_PATH = f"../{ARTIFACTS_OUTPUT_PATH}"
FEATURES_OUTPUT_PATH = f"../{FEATURES_OUTPUT_PATH}"

In [5]:
PROCESSED_RAW_FILE_PATH

'..//data\\Preprocessed_Original_ObesityDataSet.csv'

## Loading Essentials

In [6]:
# downloading the preprocessed dataset from the aws s3 bucket
if credentials["S3"] != "YOUR_S3_BUCKET_URL":
    # configuring AWS credentials
    os.environ["AWS_ACCESS_KEY_ID"] = credentials["AWS_ACCESS_KEY"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = credentials["AWS_SECRET_KEY"]

    # downloading preprocessed dataset
    s3 = boto3.client(
        "s3",
        aws_access_key_id=credentials["AWS_ACCESS_KEY"],
        aws_secret_access_key=credentials["AWS_SECRET_KEY"]
    )
    s3.download_file(
        credentials["S3"],
        PROCESSED_RAW_FILE,
        PROCESSED_RAW_FILE_PATH
    )

    # downloading artifacts from the aws s3 bucket
    !aws s3 cp --recursive s3://{credentials["S3"]}/artifacts {ARTIFACTS_OUTPUT_PATH}

    # downloading models from the aws s3 bucket
    !aws s3 cp --recursive s3://{credentials["S3"]}/features {FEATURES_OUTPUT_PATH}

Completed 1.4 KiB/4.1 KiB (402 Bytes/s) with 4 file(s) remaining
Completed 1.5 KiB/4.1 KiB (456 Bytes/s) with 4 file(s) remaining
Completed 2.4 KiB/4.1 KiB (704 Bytes/s) with 4 file(s) remaining
Completed 4.1 KiB/4.1 KiB (1.2 KiB/s) with 4 file(s) remaining  
download: s3://bucket6502-aws/artifacts/qcut_bins.pkl to ..\models\artifacts\qcut_bins.pkl
Completed 4.1 KiB/4.1 KiB (1.2 KiB/s) with 3 file(s) remaining
download: s3://bucket6502-aws/artifacts/features_encoder.pkl to ..\models\artifacts\features_encoder.pkl
Completed 4.1 KiB/4.1 KiB (1.2 KiB/s) with 2 file(s) remaining
download: s3://bucket6502-aws/artifacts/scalers.pkl to ..\models\artifacts\scalers.pkl
Completed 4.1 KiB/4.1 KiB (1.2 KiB/s) with 1 file(s) remaining
download: s3://bucket6502-aws/artifacts/label_encoder.pkl to ..\models\artifacts\label_encoder.pkl
Completed 226.0 KiB/5.2 MiB (52.4 KiB/s) with 4 file(s) remaining
download: s3://bucket6502-aws/features/y_val.pkl to ..\models\features\y_val.pkl
Completed 226.0 KiB/5.

In [23]:
# loading features
with open(os.path.join(FEATURES_OUTPUT_PATH, "X_train.pkl"), "rb") as f:
    X_train = pickle.load(f)
with open(os.path.join(FEATURES_OUTPUT_PATH, "X_val.pkl"), "rb") as f:
    X_val = pickle.load(f)
with open(os.path.join(FEATURES_OUTPUT_PATH, "y_train.pkl"), "rb") as f:
    y_train = pickle.load(f)
with open(os.path.join(FEATURES_OUTPUT_PATH, "y_val.pkl"), "rb") as f:
    y_val = pickle.load(f)


# loading artifacts
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, "qcut_bins.pkl"), "rb") as f:
    bins = pickle.load(f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, "features_encoder.pkl"), "rb") as f:
    features_encoder = pickle.load(f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, "label_encoder.pkl"), "rb") as f:
    label_encoder = pickle.load(f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, "scalers.pkl"), "rb") as f:
    scalers = pickle.load(f)

# loading feature columns
temp_df = pd.read_csv(PROCESSED_RAW_FILE_PATH, sep=",")
FEATURES_NAME = temp_df.columns.tolist()
del temp_df

In [8]:
FEATURES_NAME

['Gender_Male',
 'Age_q2',
 'Age_q3',
 'Age_q4',
 'family_history_with_overweight_yes',
 'FAVC_yes',
 'CAEC_Frequently',
 'CAEC_Sometimes',
 'CAEC_no',
 'SMOKE_yes',
 'SCC_yes',
 'CALC_Sometimes',
 'CALC_no',
 'MTRANS_Bike',
 'MTRANS_Motorbike',
 'MTRANS_Public_Transportation',
 'MTRANS_Walking',
 'INMM_1',
 'Height',
 'Weight',
 'FCVC',
 'NCP',
 'CH2O',
 'FAF',
 'TUE',
 'BMI',
 'NObeyesdad']

### Feature Selection Experimentation

In [9]:
# creating the baseline models
dt = DecisionTreeClassifier(random_state=SEED)
rf = RandomForestClassifier(random_state=SEED, verbose=0, n_jobs=-1)
xg = XGBClassifier(random_state=SEED, n_jobs=-1)
lg = LGBMClassifier(random_state=SEED, verbose=-1, objective="multiclass")
cb = CatBoostClassifier(random_seed=SEED, verbose=0, allow_writing_files=False)

In [10]:
def apply_feature_selection(
    model : Union[DecisionTreeClassifier, RandomForestClassifier, XGBClassifier, LGBMClassifier, CatBoostClassifier],
    number_features: int,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray
) -> Dict:
    # initalizing and fitting the sfs class
    sfs = SequentialFeatureSelector(
        estimator=model,
        n_features_to_select = number_features,
        cv=3
    )

    sfs.fit(X_train, y_train)

    # getting the indexes of the best features
    selected_features_idx = sfs.get_support(indices=True)

    X_train_selected = sfs.transform(X_train)
    X_val_selected = sfs.transform(X_val)

    #Train model
    model.fit(X_train_selected, y_train)

    # Caculating the validation f1 score
    y_pred_train = model.predict(X_train_selected)
    train_f1 = f1_score(y_train, y_pred_train, average="weighted")

    # Caculating the validation f1 score
    y_pred_val = model.predict(X_val_selected)
    valid_f1 = f1_score(y_val, y_pred_val, average="weighted")

    # Inferring the signature of the trained model
    signature = infer_signature(
        model_input=X_train_selected,
        model_output=y_pred_train
    )

    # saving the metrics and artifacts that we want to log in mlflow
    selected_features_names = list(map(lambda i: FEATURES_NAME[i], selected_features_idx.tolist()))

    results = {
        "train_f1": train_f1,
        "valid_f1": valid_f1,
        "features": selected_features_names,
        "model": model,
        "model_signature": signature
    }

    return results
    


In [11]:
def set_configurations_mlflow(
    model: Union[DecisionTreeClassifier, RandomForestClassifier, XGBClassifier, LGBMClassifier, CatBoostClassifier],
    y_train: np.array,
    y_val: np.array,
) -> Tuple[np.array, np.array, str, str]:
    # reshaping the target values (if needed) and setting the run name and which
    # flavor is being used for each machine learning model
    if isinstance(model, DecisionTreeClassifier):
        y_train = np.argmax(y_train, axis=1)
        y_val = np.argmax(y_val, axis=1)
        run_name = "decision_tree"
        flavor = "sklearn"
    
    if isinstance(model, RandomForestClassifier):
        run_name = "random_forest"
        flavor = "sklearn"
    
    if isinstance(model, XGBClassifier):
        run_name = "xgboost"
        flavor = "xgboost"
    
    if isinstance(model, LGBMClassifier):
        y_train = np.argmax(y_train, axis=1)
        y_val = np.argmax(y_val, axis=1)
        run_name = "lightgbm"
        flavor = "lightgbm"
    
    if isinstance(model, CatBoostClassifier):
        y_train = np.argmax(y_train, axis=1)
        y_val = np.argmax(y_val, axis=1)
        run_name = "catboost"
        flavor = "catboost"
    
    # disabling some options of the current flavor's autolog
    if flavor == "sklearn":
        mlflow.sklearn.autolog(
            log_models=False,
            log_post_training_metrics=False,
            log_model_signatures=False,
            log_input_examples=True,
            log_datasets=False,
            silent=True,
            disable=True
        )
    elif flavor == "xgboost":
        mlflow.xgboost.autolog(
            log_models=False,
            log_model_signatures=False,
            log_input_examples=True,
            log_datasets=False,
            silent=True,
            disable=True
        )
    elif flavor == "lightgbm":
        mlflow.lightgbm.autolog(
            log_models=False,
            log_model_signatures=False,
            log_input_examples=True,
            log_datasets=False,
            silent=True,
            disable=True
        )
    elif flavor == "catboost":
        # there is no autolog implemented for catboost
        pass

    return y_train, y_val, run_name, flavor

In [12]:
def run_feature_selection_experiment(
    models: List,
    min_features: int,
    max_features: int,
    experiment_id: str
) -> None:
    for model in models:
        # reshaping the target values (if needed) and setting some mlflow's configuration
        new_y_train, new_y_val, run_name, flavor = set_configurations_mlflow(
            model=model,
            y_train=y_train,
            y_val=y_val
        )
        
        # starting a new run for the current model
        with mlflow.start_run(experiment_id=experiment_id, run_name=run_name):
            pprint(f"Starting the run for the {run_name} model!\n")

            for i, n_features in enumerate(range(min_features, max_features + 1)):
                # creating a nested run inside the model's main run
                with mlflow.start_run(
                    experiment_id=experiment_id,
                    run_name=f"{run_name}_experiment_{i}",
                    nested=True
                ):
                    # running the feature selection main function
                    results = apply_feature_selection(
                        model=model,
                        number_features=n_features,
                        X_train=X_train,
                        y_train=new_y_train,
                        X_val=X_val,
                        y_val=new_y_val
                    )

                    # logging the trained model
                    if flavor == "sklearn":
                        mlflow.sklearn.log_model(
                            results["model"],
                            run_name,
                            signature=results["model_signature"]
                        )
                        # logging the model"s default parameters
                        mlflow.log_params(results["model"].get_params(deep=True))
                    elif flavor == "xgboost":
                        mlflow.xgboost.log_model(
                            results["model"],
                            run_name,
                            signature=results["model_signature"]
                        )
                        # logging the model's default parameters
                        mlflow.log_params(results["model"].get_params(deep=True))
                    elif flavor == "lightgbm":
                        mlflow.lightgbm.log_model(
                            results["model"],
                            run_name,
                            signature=results["model_signature"]
                        )
                        # logging the model's default parameters
                        mlflow.log_params(results["model"].get_params())
                    elif flavor == "catboost":
                        mlflow.catboost.log_model(
                            results["model"],
                            run_name,
                            signature=results["model_signature"]
                        )
                        # logging the model's default parameters
                        mlflow.log_params(results["model"].get_all_params())

                    # logging the training and validation scores
                    mlflow.log_metric("train_f1", results["train_f1"])
                    mlflow.log_metric("valid_f1", results["valid_f1"])

                    # logging the artifacts (original dataset, features, and encoders objects)
                    mlflow.log_artifact(PROCESSED_RAW_FILE_PATH)
                    mlflow.log_artifact(ARTIFACTS_OUTPUT_PATH)
                    mlflow.log_artifact(FEATURES_OUTPUT_PATH)

                    # logging the indexes of the best features
                    mlflow.log_param("features", results["features"])

In [13]:
# models = [dt, rf, xg, lg]
models = [lg]
min_features = math.floor(X_train.shape[1] * 0.2)
max_features = math.floor(X_train.shape[1] * 0.5)

# creating a new mlflow's experiment
experiment_id = mlflow.create_experiment(
    name=FEATURE_SELECTION_EXPERIMENT_NAME,
    tags={"version": "v1", "code_version": CODE_VERSION}
)
# experiment_id = 1
# running the feature selection experiments
run_feature_selection_experiment(
    models=models,
    min_features=min_features,
    max_features=max_features,
    experiment_id=experiment_id
)

'Starting the run for the lightgbm model!\n'


2025/10/29 01:14:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run lightgbm_experiment_0 at: http://3.90.187.253:5000/#/experiments/1/runs/1c6927a128a1439b81c30257a2a781f5
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/1


2025/10/29 01:21:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run lightgbm_experiment_1 at: http://3.90.187.253:5000/#/experiments/1/runs/ace1a8e3e650408fbaf73d718c4630b2
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/1


...

🏃 View run lightgbm_experiment_6 at: http://3.90.187.253:5000/#/experiments/1/runs/dcc5c4faad7c4b8585e9f4475ff85867
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/1


2025/10/29 02:18:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run lightgbm_experiment_7 at: http://3.90.187.253:5000/#/experiments/1/runs/73cd1a3de3d4498882eed6270f20c607
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/1


2025/10/29 02:33:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run lightgbm_experiment_8 at: http://3.90.187.253:5000/#/experiments/1/runs/dc727022466c46d79da06e79b0dfb83d
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/1
🏃 View run lightgbm at: http://3.90.187.253:5000/#/experiments/1/runs/fd8e1e2abb9144d1ba3e60057391fcde
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/1


### Hyperparameters Tuning

In [14]:
class Objective:
    def __init__(
        self,
        run_name: str,
        experiment_id: str,
        X_train: np.ndarray,
        y_train: np.array,
        X_valid: np.ndarray,
        y_valid: np.array,
        indexes: List
    ) -> None:
        self.run_name = run_name
        self.experiment_id = experiment_id
        self.X_train = X_train
        self.y_train = y_train
        self.X_valid = X_valid
        self.y_valid = y_valid
        self.indexes_name = indexes
        self.indexes = [FEATURES_NAME.index(i) for i in indexes]

        if self.run_name in ["decision_tree", "lightgbm", "catboost"]:
            self.y_train = np.argmax(self.y_train, axis=1)
            self.y_valid = np.argmax(self.y_valid, axis=1)
        
        self.X_train = self.X_train[:, self.indexes]
        self.X_valid = self.X_valid[:, self.indexes]
    
    def __call__(
        self,
        trial: optuna.trial.Trial
    ) -> float:
        with mlflow.start_run(experiment_id=self.experiment_id, nested=True):
            if self.run_name == "decision_tree":
                params = {
                    "max_depth": trial.suggest_int("max_depth", 2, 32, step=2),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 8, step=1),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 6, step=1),
                    "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5, step=0.1),
                    "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 16, step=2),
                    "random_state": SEED
                }
                model = DecisionTreeClassifier(**params)
            
            if self.run_name == "random_forest":
                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
                    "max_depth": trial.suggest_int("max_depth", 10, 50),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 32),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 32),
                    "random_state": SEED,
                    "n_jobs": -1,
                }
                model = RandomForestClassifier(**params)
            
            if self.run_name == "xgboost":
                params = {
                    "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),
                    "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
                    "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
                    "random_state": SEED,
                    "n_jobs": -1,
                }
                model = XGBClassifier(**params)
            
            if self.run_name == "lightgbm":
                params = {
                    "objective": "multiclass",
                    "verbosity": -1,
                    "random_state": SEED,
                    "n_jobs": -1,
                    "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
                    "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
                    "num_leaves": trial.suggest_int("num_leaves", 2, 256),
                    "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
                    "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
                    "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
                    "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
                }
                model = LGBMClassifier(**params)
            
            if self.run_name == "catboost":
                params = {
                    "random_seed": SEED,
                    "verbose": 0,
                    "allow_writing_files": False,
                    "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.01, 0.1),
                    "depth": trial.suggest_int("depth", 1, 12),
                    "boosting_type": trial.suggest_categorical("boosting_type", ["Ordered", "Plain"]),
                    "bootstrap_type": trial.suggest_categorical(
                        "bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]
                    )
                }
                model = CatBoostClassifier(**params)
            
            model.fit(X=self.X_train, y=self.y_train)

            # calculating the training f1 score
            train_prediction = model.predict(self.X_train)
            train_f1 = f1_score(
                y_true=self.y_train,
                y_pred=train_prediction,
                average="weighted"
            )

            # calculating the validation f1 score
            valid_prediction = model.predict(self.X_valid)
            valid_f1 = f1_score(
                y_true=self.y_valid,
                y_pred=valid_prediction,
                average="weighted"
            )

            # logging the training and validation scores
            mlflow.log_metric("train_f1", train_f1)
            mlflow.log_metric("valid_f1", valid_f1)

            # inferring the signature of the trained model
            signature = infer_signature(
                model_input=self.X_train,
                model_output=train_prediction
            )

            # saving the trained model
            if self.run_name in ["decision_tree", "random_forest"]:
                # sklearn flavor
                mlflow.sklearn.log_model(
                    model,
                    self.run_name,
                    signature=signature
                )
                # logging the model"s default parameters
                mlflow.log_params(model.get_params(deep=True))
            elif self.run_name == "xgboost":
                mlflow.xgboost.log_model(
                    model,
                    self.run_name,
                    signature=signature
                )
                # logging the model's default parameters
                mlflow.log_params(model.get_params())
            elif self.run_name == "lightgbm":
                mlflow.lightgbm.log_model(
                    model,
                    self.run_name,
                    signature=signature
                )
                # logging the model's default parameters
                mlflow.log_params(model.get_params())
            elif self.run_name == "catboost":
                mlflow.catboost.log_model(
                    model,
                    self.run_name,
                    signature=signature
                )
                # logging the model's default parameters
                mlflow.log_params(model.get_all_params())

        return valid_f1

In [15]:
# creating a new mlflow's experiment
hpt_experiment_id = mlflow.create_experiment(
    name=HYPERPARAMETER_TUNING_EXPERIMENT_NAME,
    tags={"version": "v1", "code_version": CODE_VERSION}
)

### Decision tree

In [ ]:
dt_run_name = "decision_tree"
dt_features_indexes = ['Gender_Male', 'Age_q3', 'Age_q4', 'FAVC_yes', 'CAEC_no', 'SMOKE_yes', 'MTRANS_Bike', 'MTRANS_Motorbike', 'MTRANS_Public_Transportation', 'MTRANS_Walking', 'Height', 'Weight', 'BMI']

with mlflow.start_run(experiment_id=hpt_experiment_id, run_name=dt_run_name):
    objective = Objective(
        run_name=dt_run_name,
        experiment_id=hpt_experiment_id,
        X_train=X_train,
        y_train=y_train,
        X_valid=X_val,
        y_valid=y_val,
        indexes=dt_features_indexes
    )

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100)

[I 2025-10-15 10:43:35,682] A new study created in memory with name: no-name-4cad2e7f-19c6-4f86-b17b-c4753d8b346c
2025/10/15 10:43:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run agreeable-lamb-181 at: http://98.84.176.247:5000/#/experiments/3/runs/e70f63e1bbf94c4eb1e783c2bb72a4ea
🧪 View experiment at: http://98.84.176.247:5000/#/experiments/3


[I 2025-10-15 10:44:07,074] Trial 0 finished with value: 0.06417087498964363 and parameters: {'max_depth': 32, 'min_samples_split': 7, 'min_samples_leaf': 6, 'min_weight_fraction_leaf': 0.5, 'max_leaf_nodes': 2}. Best is trial 0 with value: 0.06417087498964363.
2025/10/15 10:44:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run hilarious-calf-118 at: http://98.84.176.247:5000/#/experiments/3/runs/4b5889a80cbf4451bf8ebee73539cfef
🧪 View experiment at: http://98.84.176.247:5000/#/experiments/3


[I 2025-10-15 10:44:30,118] Trial 1 finished with value: 0.22307541590190563 and parameters: {'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 4, 'min_weight_fraction_leaf': 0.1, 'max_leaf_nodes': 2}. Best is trial 1 with value: 0.22307541590190563.
2025/10/15 10:44:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run skittish-gnu-947 at: http://98.84.176.247:5000/#/experiments/3/runs/3baa367ffc5640678a56e2180d23d252
🧪 View experiment at: http://98.84.176.247:5000/#/experiments/3


[I 2025-10-15 11:28:38,938] Trial 98 finished with value: 0.846042864595392 and parameters: {'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.0, 'max_leaf_nodes': 14}. Best is trial 20 with value: 0.8495593916502013.
2025/10/15 11:28:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run thundering-mare-384 at: http://98.84.176.247:5000/#/experiments/3/runs/17034067304b4e92a840fb1495a49d88
🧪 View experiment at: http://98.84.176.247:5000/#/experiments/3


[I 2025-10-15 11:29:10,886] Trial 99 finished with value: 0.8495593916502013 and parameters: {'max_depth': 12, 'min_samples_split': 5, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.0, 'max_leaf_nodes': 16}. Best is trial 20 with value: 0.8495593916502013.


🏃 View run decision_tree at: http://98.84.176.247:5000/#/experiments/3/runs/a43d1758aca04466a99af4a11f770205
🧪 View experiment at: http://98.84.176.247:5000/#/experiments/3


### Random Forest

In [ ]:
rf_run_name = "random_forest"
rf_features_indexes = ['Gender_Male', 'Age_q3', 'Age_q4', 'FAVC_yes', 'CALC_no', 'MTRANS_Public_Transportation', 'MTRANS_Walking', 'INMM_1', 'Height', 'Weight', 'NCP', 'CH2O', 'BMI']

with mlflow.start_run(experiment_id=hpt_experiment_id, run_name=rf_run_name):
    objective = Objective(
        run_name=rf_run_name,
        experiment_id=hpt_experiment_id,
        X_train=X_train,
        y_train=y_train,
        X_valid=X_val,
        y_valid=y_val,
        indexes=rf_features_indexes
    )

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100)



### XGBoost

In [ ]:
xgb_run_name = "xgboost"
xg_features_indexes = ['Gender_Male', 'Age_q2', 'Age_q3', 'Age_q4', 'CAEC_no', 'SMOKE_yes', 'SCC_yes', 'CALC_Sometimes', 'CALC_no', 'MTRANS_Bike', 'INMM_1', 'Weight', 'BMI']

with mlflow.start_run(experiment_id=hpt_experiment_id, run_name=xgb_run_name):
    objective = Objective(
        run_name=xgb_run_name,
        experiment_id=hpt_experiment_id,
        X_train=X_train,
        y_train=y_train,
        X_valid=X_val,
        y_valid=y_val,
        indexes=xg_features_indexes
    )

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100)



### LightGBM

In [18]:
lg_run_name = "lightgbm"
lg_features_indexes = 	['Gender_Male', 'Age_q3', 'Age_q4', 'FAVC_yes', 'CALC_Sometimes', 'CALC_no', 'Weight', 'FCVC', 'NCP', 'FAF', 'TUE', 'BMI']

with mlflow.start_run(experiment_id=hpt_experiment_id, run_name=lg_run_name):
    objective = Objective(
        run_name=lg_run_name,
        experiment_id=hpt_experiment_id,
        X_train=X_train,
        y_train=y_train,
        X_valid=X_val,
        y_valid=y_val,
        indexes=lg_features_indexes
    )

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=7)

[I 2025-10-29 02:38:58,703] A new study created in memory with name: no-name-cf3d757b-8a7b-4c49-9645-ae0bf7859778
2025/10/29 02:39:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run unequaled-stoat-242 at: http://3.90.187.253:5000/#/experiments/2/runs/3b09ea6377e747798ee3ae7c6d1a8532
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/2


[I 2025-10-29 02:39:25,455] Trial 0 finished with value: 0.8978116290476237 and parameters: {'lambda_l1': 2.332834541122989, 'lambda_l2': 0.004680794708177346, 'num_leaves': 28, 'feature_fraction': 0.9979910004671942, 'bagging_fraction': 0.6427433947130811, 'bagging_freq': 6, 'min_child_samples': 57}. Best is trial 0 with value: 0.8978116290476237.
2025/10/29 02:39:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run whimsical-asp-662 at: http://3.90.187.253:5000/#/experiments/2/runs/c714df951871476da09017c1d8a34cd9
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/2


[I 2025-10-29 02:39:53,105] Trial 1 finished with value: 0.8936318232193454 and parameters: {'lambda_l1': 0.496835501451337, 'lambda_l2': 0.0011166911668201208, 'num_leaves': 82, 'feature_fraction': 0.8496901065583219, 'bagging_fraction': 0.5543062963103953, 'bagging_freq': 5, 'min_child_samples': 55}. Best is trial 0 with value: 0.8978116290476237.
2025/10/29 02:40:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run redolent-foal-146 at: http://3.90.187.253:5000/#/experiments/2/runs/ce09948270674cec8534b97f6c69222d
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/2


[I 2025-10-29 02:41:46,392] Trial 5 finished with value: 0.8934282286034878 and parameters: {'lambda_l1': 7.037498884617964e-07, 'lambda_l2': 0.002855139998437112, 'num_leaves': 221, 'feature_fraction': 0.9837386682584819, 'bagging_fraction': 0.8245477331943809, 'bagging_freq': 2, 'min_child_samples': 20}. Best is trial 4 with value: 0.9011647658162968.
2025/10/29 02:41:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run learned-goose-291 at: http://3.90.187.253:5000/#/experiments/2/runs/d6cca910122746ee879da29c0cf4b2e2
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/2


[I 2025-10-29 02:42:14,895] Trial 6 finished with value: 0.8935118839225551 and parameters: {'lambda_l1': 8.367562897831582e-07, 'lambda_l2': 0.00041745862083066807, 'num_leaves': 222, 'feature_fraction': 0.7754235047208826, 'bagging_fraction': 0.8307443939805643, 'bagging_freq': 2, 'min_child_samples': 69}. Best is trial 4 with value: 0.9011647658162968.


🏃 View run lightgbm at: http://3.90.187.253:5000/#/experiments/2/runs/9f8eb64764cd40a6b5133037686de7d5
🧪 View experiment at: http://3.90.187.253:5000/#/experiments/2


### Cat Boost

In [ ]:
cb_run_name = "catboost"
cb_features_indexes = ['Gender_Male', 'Age_q3', 'Age_q4', 'FAVC_yes', 'CALC_no', 'MTRANS_Public_Transportation', 'INMM_1', 'Weight', 'NCP', 'CH2O', 'FAF', 'TUE', 'BMI']

with mlflow.start_run(experiment_id=hpt_experiment_id, run_name=cb_run_name):
    objective = Objective(
        run_name=cb_run_name,
        experiment_id=hpt_experiment_id,
        X_train=X_train,
        y_train=y_train,
        X_valid=X_val,
        y_valid=y_val,
        indexes=cb_features_indexes
    )

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100)

In [ ]:
# removing downloaded dataset from local
os.remove(PROCESSED_RAW_FILE_PATH)

# removing the local artifacts and features
shutil.rmtree(ARTIFACTS_OUTPUT_PATH)
shutil.rmtree(FEATURES_OUTPUT_PATH)

### Registering Best Models

In [ ]:
run_id = "3faa3a1014f046e2986e818ad12d7a1b"
run_name = "lightbgm_cls"
name = "lightgbm"
tags = {"version": "2.0", "type": "baseline", "model": name}

result = mlflow.register_model(
    # model_uri=f"runs:/{run_id}/{run_name}",
    model_uri=f"models:/m-c68776e2dae940799e63d0877cb85b6d",
    name=name,
    tags=tags,
)

Registered model 'lightgbm' already exists. Creating a new version of this model...
2025/10/15 13:03:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 150 seconds for model version to finish creation. Model name: lightgbm, version 2
Created version '2' of model 'lightgbm'.


Model loaded successfully!


In [31]:
train_f1

0.9011647658162968

0.9200032536766182